**Nombre de Proyecto:** Proyecto EdTech Learning Analytics Parte 3

**Numero de grupo:** 1

**Integrante a cargo:** Gabriela Michelle Garcia Perez

**Carnet:** 038425

**Fecha de creacion:** 15/09/2026

**Version de libro:** 1.3

In [ ]:
# ==========================================
# PROYECTO EDTECH LEARNING ANALYTICS - PARTE 3
#ARQUITECTURA DE BIG DATA CON SPARK
# ==========================================

%pip install pyspark
print("PySpark instalado ----------------")

PySpark instalado ----------------


In [ ]:
#IMPORTAMOS LIBRERIAS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
import pandas as pd
import shutil, os

print("Librerias importadas --------- ")

Librerias importadas --------- 


In [ ]:
#CREAR SESION DE SPARK
spark = SparkSession.builder \
  .master("local[*]")\
  .appName("EdTech_LearningAnalytics_Guia03_Gabriela_Garcia")\
  .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Paralelismo: {spark.sparkContext.defaultParallelism}")

Spark version: 4.0.4
Paralelismo: 2


In [ ]:
#DEFINIR RUTAS
BASE_DIR = Path("/content")
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
DATA_ANALYTICS = BASE_DIR/"datos"/"analiticos"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ANALYTICS.mkdir(parents=True, exist_ok=True)

NOMBRE_CSV = "dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv"

print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos analiticos: {DATA_ANALYTICS}")


Datos procesados: /content/datos/procesados
Datos analiticos: /content/datos/analiticos


In [ ]:
#SUBIR EL CSV A COLAB Y COLOCARLO EN LA CARPETA DE PROCESADOS
import shutil
from google.colab import files

ruta_entrada = DATA_PROCESSED/NOMBRE_CSV

forzar_subida = True
if forzar_subida and ruta_entrada.exists():
    ruta_entrada.unlink()

if not ruta_entrada.exists():
    subidos = files.upload()         
    nombre_subido = list(subidos.keys())[0]
    shutil.move(nombre_subido, ruta_entrada)
    print("CSV subido y movido --------")
else:
    print("El CSV ya estaba en la carpeta --------")

print(f"Archivo listo en: {ruta_entrada}")

Saving dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv to dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
CSV subido y movido --------
Archivo listo en: /content/datos/procesados/dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv


In [ ]:
#CARGAR DATASET LIMPIO EN SPARK

print(f"Leyendo: {ruta_entrada}")

# 'historial_cualitativo' guarda texto tipo JSON con comas dentro,
# por eso se necesita quote/escape/multiLine para que Spark no corte mal las filas
df = spark.read\
  .option("header", True)\
  .option("inferSchema", True)\
  .option("quote", '"')\
  .option("escape", '"')\
  .option("multiLine", True)\
  .csv(str(ruta_entrada))

print("Dataset cargado en Spark ------")
print(f"Filas: {df.count()}")
print(f"Columnas: {len(df.columns)}")

Leyendo: /content/datos/procesados/dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
Dataset cargado en Spark ------
Filas: 1000
Columnas: 43


In [ ]:
#VER EL ESQUEMA - MOSTRAR LA ESTRUCTURA DEL DATASET
print("Esquema de datos: -------------")
df.printSchema() #Con pandas es con .info

print("\nPrimeras filas: ")
df.select("carnet", "materia", "nota_teoria", "nota_laboratorio",
          "nota_final", "riesgo_academico", "probabilidad_desercion")

Esquema de datos: -------------
root
 |-- carnet: string (nullable = true)
 |-- nombres: string (nullable = true)
 |-- apellidos: string (nullable = true)
 |-- fecha_nacimiento: date (nullable = true)
 |-- edad: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- facultad: string (nullable = true)
 |-- carrera: string (nullable = true)
 |-- anio_ingreso: integer (nullable = true)
 |-- ciclo_actual: integer (nullable = true)
 |-- turno: string (nullable = true)
 |-- municipio_residencia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- telefono: integer (nullable = true)
 |-- correo_institucional: string (nullable = true)
 |-- acceso_internet_casa: string (nullable = true)
 |-- trabaja: string (nullable = true)
 |-- horas_trabajo_semana: double (nullable = true)
 |-- beca: string (nullable = true)
 |-- ciclo: integer (nullable = true)
 |-- materia: string (nullable = true)
 |-- nota_teoria: double (nullable = true)
 |-- nota_laboratorio: double (nullable 

DataFrame[carnet: string, materia: string, nota_teoria: double, nota_laboratorio: double, nota_final: double, riesgo_academico: string, probabilidad_desercion: double]

In [ ]:
#PARTICIONES CON SPARK
print(f"\nParticiones iniciales: {df.rdd.getNumPartitions()}")



Particiones iniciales: 1


In [ ]:
#FILTRAR REGISTROS VALIDOS
df_validos = df.filter(
    (F.col("nota_final") > 0) &
    F.col("materia").isNotNull() &
    F.col("riesgo_academico").isNotNull()
)

print(f"Registros despues de filtrar: {df_validos.count()}")
print(f"Registros eliminados: {df.count() - df_validos.count()}")


Registros despues de filtrar: 1000
Registros eliminados: 0


In [ ]:
#AQUI PODEMOS VER QUE LA PARTE 2 ESTA CORRECTAMENTE EJECUTADA, YA QUE LOS DATOS YA VENIAN LIMPIOS

In [ ]:
#CREAR COLUMNA DERIVADA -- MARCAR ESTUDIANTES EN RIESGO ALTO

df_enriquecido = df_validos.withColumn(
    "en_riesgo_alto",
    F.when(F.col("riesgo_academico") == "Alto", True)
    .otherwise(False)
)

print("Columna 'en_riesgo_alto' creada --------------")
df_enriquecido.select("carnet", "materia", "riesgo_academico", "en_riesgo_alto").show(5)

Columna 'en_riesgo_alto' creada --------------
+-------+--------------+----------------+--------------+
| carnet|       materia|riesgo_academico|en_riesgo_alto|
+-------+--------------+----------------+--------------+
|RR19001|        Ingles|           Medio|         false|
|RF24002|   Estadistica|           Medio|         false|
|FG23003|      Fisica I|            Bajo|         false|
|RL21004|        Ingles|           Medio|         false|
|PM21005|Programacion I|           Medio|         false|
+-------+--------------+----------------+--------------+
only showing top 5 rows


In [ ]:
#AGREGAR POR MATERIA

resumen_materia = df_enriquecido.groupBy("materia").agg(
    F.round(F.avg("nota_final"), 2).alias("promedio_nota_final"),
    F.count("*").alias("estudiantes"),
    F.sum(F.when(F.col("en_riesgo_alto"), 1).otherwise(0)).alias("estudiantes_riesgo_alto")
).orderBy(F.desc("promedio_nota_final"))

print("Promedio academico por curso: -------")
resumen_materia.show(truncate=False)

Promedio academico por curso: -------
+--------------+-------------------+-----------+-----------------------+
|materia       |promedio_nota_final|estudiantes|estudiantes_riesgo_alto|
+--------------+-------------------+-----------+-----------------------+
|Estadistica   |7.23               |169        |69                     |
|Programacion I|7.08               |178        |88                     |
|Base de Datos |7.04               |164        |76                     |
|Fisica I      |7.03               |154        |76                     |
|Matematica I  |7.02               |167        |69                     |
|Ingles        |6.97               |168        |89                     |
+--------------+-------------------+-----------+-----------------------+



In [ ]:
#CREAR VISTA TEMPORAL
df_enriquecido.createOrReplaceTempView("rendimiento_academico")
print("Vista temporal 'rendimiento_academico' creada")


Vista temporal 'rendimiento_academico' creada


In [ ]:
#CREAR CONSULTA SQL - PROMEDIO Y RIESGO POR MATERIA

consulta = """
  SELECT
    materia,
    COUNT(*) AS estudiantes,
    ROUND(AVG(nota_final), 2) AS promedio_nota_final,
    SUM(CASE WHEN riesgo_academico = 'Alto' THEN 1 ELSE 0 END) AS estudiantes_riesgo_alto,
    ROUND(100 * SUM(CASE WHEN riesgo_academico = 'Alto' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_riesgo_alto
  FROM rendimiento_academico
  GROUP BY materia
  ORDER BY promedio_nota_final DESC
"""

resultado_sql = spark.sql(consulta)
print("Consulta SQL - PROMEDIO Y RIESGO POR MATERIA")
resultado_sql.show(truncate=False)

Consulta SQL - PROMEDIO Y RIESGO POR MATERIA
+--------------+-----------+-------------------+-----------------------+---------------+
|materia       |estudiantes|promedio_nota_final|estudiantes_riesgo_alto|pct_riesgo_alto|
+--------------+-----------+-------------------+-----------------------+---------------+
|Estadistica   |169        |7.23               |69                     |40.8           |
|Programacion I|178        |7.08               |88                     |49.4           |
|Base de Datos |164        |7.04               |76                     |46.3           |
|Fisica I      |154        |7.03               |76                     |49.4           |
|Matematica I  |167        |7.02               |69                     |41.3           |
|Ingles        |168        |6.97               |89                     |53.0           |
+--------------+-----------+-------------------+-----------------------+---------------+



In [ ]:
#VER PLAN DE EJECUCION
print("\nPLAN DE EJECUCION FORMATEADO: ------------")
resultado_sql.explain("formatted")


PLAN DE EJECUCION FORMATEADO: ------------
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan csv  (1)


(1) Scan csv 
Output [3]: [materia#110, nota_final#113, riesgo_academico#130]
Batched: false
Location: InMemoryFileIndex [file:/content/datos/procesados/dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv]
PushedFilters: [IsNotNull(nota_final), GreaterThan(nota_final,0.0), IsNotNull(materia), IsNotNull(riesgo_academico)]
ReadSchema: struct<materia:string,nota_final:double,riesgo_academico:string>

(2) Filter
Input [3]: [materia#110, nota_final#113, riesgo_academico#130]
Condition : (((isnotnull(nota_final#113) AND (nota_final#113 > 0.0)) AND isnotnull(materia#110)) AND isnotnull(riesgo_academico#130))

(3) HashAggregate
Input [3]: [materia#110, nota_final#113, riesgo_academico#130]
Keys [1]: [materia#110]
Functions [3]: 

In [ ]:
#INSPECCIONAR PARTICIONES ACTUALES
print(f"Particiones actuales: {df_enriquecido.rdd.getNumPartitions()}")


Particiones actuales: 1


In [ ]:
#REPARTICIONAR EN 4
df_particionado = df_enriquecido.repartition(4)
print(f"Nuevas particiones: {df_particionado.rdd.getNumPartitions()}")

Nuevas particiones: 4


In [ ]:
#GUARDAR EN PARQUET
ruta_parquet = DATA_ANALYTICS/"rendimiento_academico_parquet"

df_particionado.write \
  .mode("overwrite")\
  .parquet(str(ruta_parquet))

print(f"Parquet guardado en: {ruta_parquet}")

Parquet guardado en: /content/datos/analiticos/rendimiento_academico_parquet


In [ ]:
#LEER PARQUET
df_leido = spark.read.parquet(str(ruta_parquet))

print(f"Filas leidas: {df_leido.count()}")
print("Primeras filas: ------------------")
df_leido.select("carnet", "materia", "nota_final", "riesgo_academico", "en_riesgo_alto").show(5, truncate=False)

Filas leidas: 1000
Primeras filas: ------------------
+-------+--------------+----------+----------------+--------------+
|carnet |materia       |nota_final|riesgo_academico|en_riesgo_alto|
+-------+--------------+----------+----------------+--------------+
|CD18438|Base de Datos |6.76      |Medio           |false         |
|ÁV20981|Fisica I      |7.13      |Alto            |true          |
|VP20044|Matematica I  |8.26      |Bajo            |false         |
|RC23277|Programacion I|8.44      |Bajo            |false         |
|FF23225|Fisica I      |6.74      |Medio           |false         |
+-------+--------------+----------+----------------+--------------+
only showing top 5 rows


In [ ]:
#AGREGACIONES CON PANDAS

df_pd = pd.read_csv(ruta_entrada)
df_pd = df_pd[(df_pd["nota_final"] > 0) & df_pd["materia"].notna() & df_pd["riesgo_academico"].notna()]

pd_materia = df_pd.groupby("materia")["nota_final"].mean().round(2).reset_index()
pd_materia.columns = ["materia", "promedio_pandas"]

# Tambien se calcula el % de riesgo alto con pandas para compararlo con Spark
pd_riesgo = (df_pd.groupby("materia")["riesgo_academico"]
             .apply(lambda s: round(100 * (s == "Alto").mean(), 1))
             .reset_index())
pd_riesgo.columns = ["materia", "pct_riesgo_pandas"]
pd_materia = pd_materia.merge(pd_riesgo, on="materia")

print("Pandas - promedio de nota final y % riesgo alto por materia")
print(pd_materia)

Pandas - promedio de nota final y % riesgo alto por materia
          materia  promedio_pandas  pct_riesgo_pandas
0   Base de Datos             7.04               46.3
1     Estadistica             7.23               40.8
2        Fisica I             7.03               49.4
3          Ingles             6.97               53.0
4    Matematica I             7.02               41.3
5  Programacion I             7.08               49.4


In [ ]:
#RESULTADO SPARK (AGREGACIONES) PARA COMPARAR CON PANDAS

# Se agrega el % de riesgo alto de Spark (calculado desde estudiantes_riesgo_alto / estudiantes)
sp_materia = resumen_materia.select(
    "materia", "promedio_nota_final",
    F.round(100 * F.col("estudiantes_riesgo_alto") / F.col("estudiantes"), 1).alias("pct_riesgo_alto")
).orderBy("materia").toPandas()
sp_materia.columns = ["materia", "promedio_spark", "pct_riesgo_spark"]

print("\nSPARK - PROMEDIO DE NOTA FINAL Y % RIESGO ALTO POR MATERIA")
print(sp_materia)



SPARK - PROMEDIO DE NOTA FINAL Y % RIESGO ALTO POR MATERIA
          materia  promedio_spark  pct_riesgo_spark
0   Base de Datos            7.04              46.3
1     Estadistica            7.23              40.8
2        Fisica I            7.03              49.4
3          Ingles            6.97              53.0
4    Matematica I            7.02              41.3
5  Programacion I            7.08              49.4


In [ ]:
#COMPARAR RESULTADOS
print("\nComparacion Pandas vs Spark")

comparacion = pd.merge(pd_materia, sp_materia, on="materia")
comparacion["diferencia"] = (comparacion["promedio_pandas"] - comparacion["promedio_spark"]).abs().round(6)
# Tambien se compara el % de riesgo alto
comparacion["dif_riesgo"] = (comparacion["pct_riesgo_pandas"] - comparacion["pct_riesgo_spark"]).abs().round(6)
print(comparacion)

# La condicion revisa promedio Y riesgo
if comparacion["diferencia"].max() < 0.001 and comparacion["dif_riesgo"].max() < 0.001:
    print("\nLos resultados son consistentes entre Pandas y Spark --------- ")
else:
    print("\nHay diferencias, revisar filtros aplicados")


Comparacion Pandas vs Spark
          materia  promedio_pandas  pct_riesgo_pandas  promedio_spark  \
0   Base de Datos             7.04               46.3            7.04   
1     Estadistica             7.23               40.8            7.23   
2        Fisica I             7.03               49.4            7.03   
3          Ingles             6.97               53.0            6.97   
4    Matematica I             7.02               41.3            7.02   
5  Programacion I             7.08               49.4            7.08   

   pct_riesgo_spark  diferencia  dif_riesgo  
0              46.3         0.0         0.0  
1              40.8         0.0         0.0  
2              49.4         0.0         0.0  
3              53.0         0.0         0.0  
4              41.3         0.0         0.0  
5              49.4         0.0         0.0  

Los resultados son consistentes entre Pandas y Spark --------- 


In [ ]:
#REPORTE DE ARQUITECTURA
#Se calculan de acuerdo a los resultados para que siempre coincidan.
mejor_prom = resumen_materia.orderBy(F.desc("promedio_nota_final")).first()
pct_df = resumen_materia.withColumn(
    "pct_riesgo_alto", F.round(100 * F.col("estudiantes_riesgo_alto") / F.col("estudiantes"), 1)
)
mayor_riesgo = pct_df.orderBy(F.desc("pct_riesgo_alto")).first()

#REPORTE DE ARQUITECTURA
print("\nREPORTE DE ARQUITECTURA - PROYECTO FINAL EDTECH")
print("=" * 55)
print(f"Versión de Spark: {spark.version}")
print(f"Dataset de entrada: {NOMBRE_CSV}")
print(f"Filas procesadas: {df_enriquecido.count()}")
print(f"Columnas originales: {len(df.columns)}")
print(f"Columnas finales: {len(df_particionado.columns)}")
print(f"Particiones iniciales: 1")
print(f"Particiones finales: {df_particionado.rdd.getNumPartitions()}")
print(f"Vista temporal SQL: rendimiento_academico")
print(f"Salida Parquet: {ruta_parquet}")
print("Indicador de riesgo usado: riesgo_academico")
print(f"Materia con mejor promedio: {mejor_prom['materia']} ({mejor_prom['promedio_nota_final']})")
print(f"Materia con mayor % de riesgo alto: {mayor_riesgo['materia']} ({mayor_riesgo['pct_riesgo_alto']}%)")
print("=" * 55)


REPORTE DE ARQUITECTURA - PROYECTO FINAL EDTECH
Versión de Spark: 4.0.4
Dataset de entrada: dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
Filas procesadas: 1000
Columnas originales: 43
Columnas finales: 44
Particiones iniciales: 1
Particiones finales: 4
Vista temporal SQL: rendimiento_academico
Salida Parquet: /content/datos/analiticos/rendimiento_academico_parquet
Indicador de riesgo usado: riesgo_academico
Materia con mejor promedio: Estadistica (7.23)
Materia con mayor % de riesgo alto: Ingles (53.0%)


In [ ]:
#CERRAR SESION
spark.stop()
print("Sesion de Spark finalizada!!!!!")

Sesion de Spark finalizada!!!!!


In [ ]:
#DESCARGAR TODOS LOS ARCHIVOS GENERADOS
!zip -r /content/TerceraParte_semana3.zip /content/datos/

from google.colab import files
files.download("/content/TerceraParte_semana3.zip")

print("Todos los archivos descargados!")

updating: content/datos/ (stored 0%)
updating: content/datos/procesados/ (stored 0%)
updating: content/datos/procesados/dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv (deflated 90%)
updating: content/datos/analiticos/ (stored 0%)
updating: content/datos/analiticos/rendimiento_academico_parquet/ (stored 0%)
updating: content/datos/analiticos/rendimiento_academico_parquet/._SUCCESS.crc (stored 0%)
updating: content/datos/analiticos/rendimiento_academico_parquet/.part-00000-959f89af-8f50-4b91-8029-b44c65799658-c000.snappy.parquet.crc (stored 0%)
updating: content/datos/analiticos/rendimiento_academico_parquet/.part-00001-959f89af-8f50-4b91-8029-b44c65799658-c000.snappy.parquet.crc (stored 0%)
updating: content/datos/analiticos/rendimiento_academico_parquet/part-00001-959f89af-8f50-4b91-8029-b44c65799658-c000.snappy.parquet (deflated 33%)
updating: content/datos/analiticos/rendimiento_academico_parquet/_SUCCESS (stored 0%)
updating: content/datos/analiticos/rendimiento_academico_par

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Todos los archivos descargados!
